This is a dataset of regional US cuisine that has been scraped from the "List of Regional Dishes of the United States" Wikipedia page (https://en.wikipedia.org/wiki/List_of_regional_dishes_of_the_United_States#City_chicken). Scraped with beautiful soup, (sorta) standardized with Python, and then manually organized to account for hard to deal with data. 

In [3]:
import pandas as pd 
import geopandas as gpd

foods_csv = '/Users/benmccarthy/Development/data_projects/regional_foods/regional_foods_EDITED_FOR_PROJ_4.csv'
foods_df = pd.read_csv(foods_csv)


states_geojson = '/Users/benmccarthy/Development/data_projects/regional_foods/states_geojson.json'
states_gdf = gpd.read_file(states_geojson, engine="pyogrio")
# states_gdf

# foods_df

geographies = []
# iterate thorugh foods, clean data, 
for index, row in foods_df.iterrows():
    # use eval to replace nasty little weird ass quotes
    
    state_list = eval(row['state'].replace("’","'").replace("‘","'").replace('"',"'").replace('”',"'").replace('“',"'"))
    # logic to handle rows without states (only regions, multiple regions, sub-regions) for later date. 
    # if len(state_list) == 0:
    #     print(row['region'])
    #     if row['region'] == 'Multiple':
    #         print(row['sub-region'])
    for state in state_list:
        # how does this filter work? 
        # we are looping through foods, with a nested loop looping through the states array of foods. 
        # state is the state from foods
        # if we find a match in states_gdf (states and geometries gdf) we create an obj with state name and geometry ->
        # and spread the rest of the row from foods_df to the new obj. 
        search = states_gdf[states_gdf['NAME'] == state]
        # else branch for when there is no state or states, replace with states from region?
        # print(len(search))
        if len(search):
            # append to geographies array, which we will turn into our own new df
            
            geographies.append({
                'geometry': search.iloc[0].geometry,
                'state' : state,
                # spread rows
                **row
            })
       


In [4]:
# foods_with_geographies_gpd = gpd.GeoDataFrame(geographies, crs='4326')
# foods_with_geographies_gpd
# geographies_gpd.to_file("/Users/benmccarthy/Development/data_projects/Jupyter.geojson", driver="GeoJSON")

This gives me a gdf of many geometries of states, each with a corresponding food. But for my leaflet app, where I want a user to be able to click a state and get a list of foods from that state, I need to condense the foods into the (less than 50) states. How do I do this?

for each row in the foods_with_geographies gpd, i need to loop through the states within the states column, and add each food (row) to an existing state in array format within an obj. 

Is it best to start over and do this in reverse from how I did it before? Taking the states_gdf and appending food info to it in a loop? 

In [15]:
import ast

def parse_list(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        val = val.replace("’","'").replace("‘","'").replace('"',"'").replace('”',"'").replace('“',"'")
        return ast.literal_eval(val)
    return []

# this should have one state for every state in the states_geographies gdf
# create a new column that is an array of food properties
states_with_foods_obj_array = []
# loop through states with geographies
for index, state_with_geography in states_gdf.iterrows():
    # print(state_with_geography)
    # for every state_w_geo loop through foods. Check state column, loop through state array.
    # if food state matches geo_state, append to a new array within the state object
    new_state_with_geometry_and_foods_array = {
                    **state_with_geography,
                    'foods_array' : []
                }
    state_name_with_geography = state_with_geography['NAME']
    for index, food_row in foods_df.iterrows():
        state_name_array_from_foods = eval(food_row['state'].replace("’","'").replace("‘","'").replace('"',"'").replace('”',"'").replace('“',"'"))
        for state_name_from_food in state_name_array_from_foods:
            # print(state_name_from_food)
            # if state_name_from_food == state_name_with_geography, append food info obj to state obj
            if (state_name_from_food == state_name_with_geography):
                # print(food_row.state, state_name_with_geography)
                # print(state_with_geography, food_row)
                # create an object from state_geographies, with a food key with arr val that contains food_row objs
                # food_row_obj (type, dish, region, state, city, sub-region, location, description, image_url, citations)
                # state_geometry obj spread? where should this obj live? in top loop
                
                # image_url_modified = None
                # if food_row.image_url
                new_food_obj = {
                    'type': food_row.type,
                    'dish' : food_row.dish,
                    'region' : food_row.region,
                    'state' : parse_list(food_row.state),
                    'city' : parse_list(food_row.city),
                    'sub-region' : parse_list(food_row['sub-region']),
                    'location' : food_row.location,
                    'description' : food_row.description,
                    'image_url' : food_row.image_url
                }
                # append new object to obj within state/geo loop
                new_state_with_geometry_and_foods_array['foods_array'].append(new_food_obj)
    # append to main array (states_with_foods_obj_array)
    states_with_foods_obj_array.append(new_state_with_geometry_and_foods_array)

         
states_with_geographies_and_foods_gdf = gpd.GeoDataFrame(states_with_foods_obj_array, crs='4326')
# states_with_geographies_and_foods_gdf.iloc[6].foods_array
states_with_geographies_and_foods_gdf

,GEO_ID,STATE,NAME,LSAD,CENSUSAREA,geometry,foods_array
0,0400000US23,23,Maine,,30842.923,"MULTIPOLYGON (((-67.61976 44.51975, -67.61541 ...","[{'type': 'Desserts and confectionery', 'dish'..."
1,0400000US25,25,Massachusetts,,7800.058,"MULTIPOLYGON (((-70.83204 41.60650, -70.82373 ...","[{'type': 'General Cuisine', 'dish': 'Boston b..."
2,0400000US26,26,Michigan,,56538.901,"MULTIPOLYGON (((-88.68443 48.11579, -88.67563 ...","[{'type': 'General Cuisine', 'dish': 'City chi..."
3,0400000US30,30,Montana,,145545.801,"POLYGON ((-104.05770 44.99743, -104.25015 44.9...",[]
4,0400000US32,32,Nevada,,109781.180,"POLYGON ((-114.05060 37.00040, -114.04999 36.9...",[]
5,0400000US34,34,New Jersey,,7354.220,"POLYGON ((-75.52684 39.65571, -75.52634 39.656...","[{'type': 'General Cuisine', 'dish': 'Pork rol..."
6,0400000US36,36,New York,,47126.399,"MULTIPOLYGON (((-71.94356 41.28668, -71.92680 ...","[{'type': 'General Cuisine', 'dish': 'Baked zi..."
7,0400000US37,37,North Carolina,,48617.905,"MULTIPOLYGON (((-82.60288 36.03983, -82.60074 ...","[{'type': 'General Cuisine', 'dish': 'Livermus..."
8,0400000US39,39,Ohio,,40860.694,"MULTIPOLYGON (((-82.81349 41.72347, -82.81049 ...","[{'type': 'General Cuisine', 'dish': 'Cincinna..."
9,0400000US42,42,Pennsylvania,,44742.703,"POLYGON ((-75.41504 39.80179, -75.42804 39.809...","[{'type': 'General Cuisine', 'dish': 'City chi..."


In [16]:
# states_with_geographies_and_foods_gdf.to_file("/Users/benmccarthy/Development/data_projects/States_with_foods_5.geojson", driver="GeoJSON", engine="pyogrio")

In [48]:
# 
states_with_dishes = []
for index, state_with_foods in states_with_geographies_and_foods_gdf.iterrows():
    
    # print(state_with_foods['NAME'], len(states_with_geographies_and_foods_gdf['foods_array'].iloc[index])) 
    name = state_with_foods['NAME']
    count = len(states_with_geographies_and_foods_gdf['foods_array'].iloc[index])
    count_obj = {
        "state" : name,
        "dishes" : count
    }
    states_with_dishes.append(count_obj)
sorted_states = sorted(states_with_dishes, key=lambda f: f['dishes'], reverse=True)

States_with_dish_count_df = pd.DataFrame(sorted_states)

States_with_dish_count_df


,state,dishes
0,New York,48
1,Louisiana,26
2,California,23
3,Pennsylvania,18
4,Illinois,15
5,Texas,14
6,Hawaii,12
7,Ohio,11
8,Missouri,10
9,New Jersey,8
